# HayabusaNet

**Paper:** HayabusaNet: Hybrid attention-based multiscale fusion CNN for accurate and efficient brain tumor classification in MRI scans  
**Authors:** Rizal Dwi Prayogo, Siti Amatullah Karimah, and Hidetaka Nambo  
**Journal:** Expert Systems With Applications, 331, 133135, 2026  
**DOI:** https://doi.org/10.1016/j.eswa.2026.133135  
**License:** CC BY 4.0  

This notebook provides the TensorFlow/Keras implementation of HayabusaNet for multiclass brain tumor MRI classification. The code defines the model architecture, training configuration, validation evaluation, confusion matrix, ROC analysis, model reporting, and inference-time measurement.

> This implementation is intended for research and reproducibility purposes only. It is not intended for clinical diagnosis or direct medical decision-making.

## GPU Memory Cleanup

In [ ]:
import gc
import tensorflow as tf

def clear_gpu_memory():
    print("Clearing GPU memory...")
    gc.collect()                       # Release Python garbage-collected objects.
    tf.keras.backend.clear_session()  # Clear the current TensorFlow/Keras session.
    print("GPU memory cleared.")

# Run memory cleanup before starting model training.
clear_gpu_memory()

## Dependencies and Runtime Configuration

This cell imports the required libraries and configures the runtime environment.

In [ ]:
# Model-building dependencies and runtime configuration.
import os, gc, sys, time
from io import StringIO
import random

import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model, regularizers
from tensorflow.keras.layers import (
    Dense,
    Activation,
    Dropout,
    GlobalAveragePooling2D,
    Input,
    Add,
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.metrics import Precision, Recall
from sklearn.metrics import (
    roc_curve,
    auc,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras.preprocessing import image_dataset_from_directory
from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2
from tensorflow.python.profiler.model_analyzer import profile
from tensorflow.python.profiler.option_builder import ProfileOptionBuilder
from tensorflow.keras import mixed_precision

# Enable mixed-precision computation to reduce GPU memory usage and improve throughput.
mixed_precision.set_global_policy('mixed_float16')

# Set a fixed seed to improve reproducibility across NumPy, Python, and TensorFlow.
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

# Configure GPU memory growth to avoid allocating all GPU memory at startup.
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        logical_gpus = tf.config.experimental.list_logical_devices('GPU')
        print(f"{len(gpus)} Physical GPUs, {len(logical_gpus)} Logical GPUs")
    except RuntimeError as e:
        print(e)

tf.config.optimizer.set_experimental_options({
    'gradient_checkpointing': True,
    'layout_optimizer': False,
    'constant_folding': False,
    'disable_meta_optimizer': True
})

## Experiment Configuration

In [ ]:
TARGET_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCH = 50
l_rate = 0.001

# This notebook assumes that the dataset has already been preprocessed and organized into:
train_dir = "dataset/train"
validation_dir = "dataset/validation"

model_name = "HayabusaNet_Multiclass"

output_dir = "outputs"
os.makedirs(output_dir, exist_ok=True)

## Dataset Loading

In [ ]:
train_generator = image_dataset_from_directory(
    train_dir,
    batch_size=BATCH_SIZE,
    image_size=TARGET_SIZE,
    shuffle=True,
    label_mode='categorical',
    seed=SEED
).prefetch(2)

validation_generator = image_dataset_from_directory(
    validation_dir,
    batch_size=BATCH_SIZE,
    image_size=TARGET_SIZE,
    shuffle=False,
    label_mode='categorical'
).prefetch(1)

## Hybrid Attention Modules

This cell implements the attention layers used in HayabusaNet, including channel attention, spatial attention, CBAM, self-attention, and the final hybrid attention block. CBAM emphasizes informative channels and spatial regions, while self-attention captures global contextual dependencies. The final hybrid attention module combines both outputs through residual-gated fusion.

In [ ]:
# Hybrid attention modules used in HayabusaNet.
class ChannelAttention(layers.Layer):
    """Channel attention module from CBAM.

    This layer recalibrates feature channels using shared MLP outputs from
    global average pooling and global max pooling descriptors.
    """

    def __init__(self, reduction_ratio=16, **kwargs):
        """Initialize the channel attention layer.

        Args:
            reduction_ratio: Reduction ratio for the hidden MLP dimension.
            **kwargs: Additional keyword arguments for the Keras Layer class.
        """
        super().__init__(**kwargs)
        self.reduction_ratio = reduction_ratio

    def build(self, input_shape):
        """Create pooling and shared MLP layers based on the input channels.

        Args:
            input_shape: Shape of the input feature map.
        """
        ch = int(input_shape[-1])
        hidden = max(ch // self.reduction_ratio, 1)

        self.gap = layers.GlobalAveragePooling2D(keepdims=True)
        self.gmp = layers.GlobalMaxPooling2D(keepdims=True)
        self.fc1 = layers.Dense(hidden, activation='relu')
        self.fc2 = layers.Dense(ch)

    def call(self, x):
        """Apply channel attention to the input feature map.

        Args:
            x: Input feature map.

        Returns:
            Channel-refined feature map.
        """
        x_dtype = x.dtype

        avg = self.gap(x)
        maxp = self.gmp(x)

        avg_mlp = self.fc2(self.fc1(tf.cast(avg, tf.float32)))
        max_mlp = self.fc2(self.fc1(tf.cast(maxp, tf.float32)))

        ca = tf.nn.sigmoid(avg_mlp + max_mlp)
        ca = tf.cast(ca, x_dtype)
        return x * ca


class SpatialAttention(layers.Layer):
    """Spatial attention module from CBAM.

    This layer emphasizes informative spatial regions using channel-wise
    average and max pooling followed by a convolutional attention map.
    """

    def __init__(self, kernel_size=7, **kwargs):
        """Initialize the spatial attention layer.

        Args:
            kernel_size: Kernel size for the spatial attention convolution.
            **kwargs: Additional keyword arguments for the Keras Layer class.
        """
        super().__init__(**kwargs)
        assert kernel_size in (3, 5, 7), "Common CBAM kernel sizes are 3, 5, or 7."
        self.kernel_size = kernel_size
        self.conv = layers.Conv2D(
            1, kernel_size=self.kernel_size, padding='same', activation='sigmoid'
        )

    def call(self, x):
        """Apply spatial attention to the input feature map.

        Args:
            x: Input feature map.

        Returns:
            Spatially refined feature map.
        """
        x_dtype = x.dtype

        avg = tf.reduce_mean(x, axis=-1, keepdims=True)
        maxp = tf.reduce_max(x, axis=-1, keepdims=True)
        concat = tf.concat([avg, maxp], axis=-1)

        sa = self.conv(tf.cast(concat, tf.float32))
        sa = tf.cast(sa, x_dtype)
        return x * sa


class CBAMBlock(layers.Layer):
    """Convolutional Block Attention Module.

    This block applies channel attention followed by spatial attention.
    """

    def __init__(self, reduction_ratio=16, spatial_kernel=7, **kwargs):
        """Initialize the CBAM block.

        Args:
            reduction_ratio: Reduction ratio for channel attention.
            spatial_kernel: Kernel size for spatial attention.
            **kwargs: Additional keyword arguments for the Keras Layer class.
        """
        super().__init__(**kwargs)
        self.ca = ChannelAttention(reduction_ratio=reduction_ratio)
        self.sa = SpatialAttention(kernel_size=spatial_kernel)

    def call(self, x):
        """Apply CBAM refinement to the input feature map.

        Args:
            x: Input feature map.

        Returns:
            Feature map refined by channel and spatial attention.
        """
        x = self.ca(x)
        x = self.sa(x)
        return x


class SelfAttention(layers.Layer):
    """Lightweight self-attention module.

    This layer models global contextual dependencies using query, key,
    and value projections on a downsampled feature map.
    """

    def __init__(self, in_dim):
        """Initialize the self-attention layer.

        Args:
            in_dim: Number of input feature channels.
        """
        super().__init__()
        self.in_dim = in_dim

        self.query_conv = layers.Conv2D(in_dim // 8, kernel_size=1)
        self.key_conv = layers.Conv2D(in_dim // 8, kernel_size=1)
        self.value_conv = layers.Conv2D(in_dim, kernel_size=1)

        self.gamma = self.add_weight(
            name='gamma',
            shape=(),
            initializer='zeros',
            trainable=True
        )

    def call(self, x):
        """Apply self-attention to the input feature map.

        Args:
            x: Input feature map.

        Returns:
            Feature map enhanced with global contextual information.
        """
        input_dtype = x.dtype

        x_down = tf.nn.avg_pool2d(x, ksize=4, strides=4, padding='SAME')
        x_down_float32 = tf.cast(x_down, tf.float32)

        q = self.query_conv(x_down_float32)
        k = self.key_conv(x_down_float32)
        v = self.value_conv(x_down_float32)

        q = tf.reshape(q, [tf.shape(x_down)[0], -1, self.in_dim // 8])
        k = tf.reshape(k, [tf.shape(x_down)[0], -1, self.in_dim // 8])
        v = tf.reshape(v, [tf.shape(x_down)[0], -1, self.in_dim])

        energy = tf.matmul(q, k, transpose_b=True)
        attention = tf.nn.softmax(energy)
        out = tf.matmul(attention, v)
        out = tf.reshape(out, tf.shape(x_down))

        out = tf.image.resize(out, tf.shape(x)[1:3])
        out = tf.cast(out, input_dtype)
        gamma = tf.cast(self.gamma, input_dtype)

        return gamma * out + x


class HybridAttention(layers.Layer):
    """Hybrid attention module used in HayabusaNet.

    This layer combines CBAM-based local refinement and self-attention-based
    global context using residual-gated fusion.
    """

    def __init__(self, in_dim, alpha=0.5):
        """Initialize the hybrid attention layer.

        Args:
            in_dim: Number of input feature channels.
            alpha: Initial value of the learnable residual fusion weight.
        """
        super().__init__()
        self.cbam = CBAMBlock()
        self.self_attn = SelfAttention(in_dim)

        self.alpha = self.add_weight(
            name="alpha",
            shape=(),
            initializer=tf.keras.initializers.Constant(alpha),
            trainable=True,
            dtype=tf.float32
        )

    def call(self, x):
        """Apply hybrid attention to the input feature map.

        Args:
            x: Input feature map.

        Returns:
            Feature map refined by residual-gated hybrid attention.
        """
        cbam_out = self.cbam(x)
        sa_out = self.self_attn(x)

        x_dtype = x.dtype
        cbam_out = tf.cast(cbam_out, x_dtype)
        sa_out = tf.cast(sa_out, x_dtype)
        alpha = tf.cast(self.alpha, x_dtype)

        return x + alpha * (cbam_out + sa_out)

        # Ablation alternative: additive fusion.
        # return cbam_out + sa_out

        # Ablation alternative: concatenation-based fusion.
        # return tf.concat([cbam_out, sa_out], axis=-1)

        # Ablation alternative: multiplicative fusion.
        # return cbam_out * sa_out

## HayabusaNet Model Definition and Compilation

This cell defines the HayabusaNet architecture for multiclass brain tumor MRI classification. The model uses three multiscale depthwise-separable convolution branches with 3×3, 5×5, and 7×7 kernels, element-wise feature fusion, post-fusion hybrid attention, and a fully connected classification head.

In [ ]:
def branch(x, k, depth, base_filters=32):
    """Build one multiscale convolutional branch.

    Args:
        x: Input feature map.
        k: Kernel size used in the separable convolution layers.
        depth: Number of repeated convolution-pooling blocks.
        base_filters: Number of filters in the first convolutional block.

    Returns:
        Output feature map from one multiscale branch.
    """
    c = x
    for i in range(depth):
        f = base_filters * (2**i)
        c = layers.SeparableConv2D(f, (k, k), padding='same', strides=1, use_bias=False)(c)
        c = layers.Activation('relu')(c)
        c = layers.MaxPooling2D(2, padding='same')(c)

    # Ablation option: apply attention inside each branch.
    # c = HybridAttention(in_dim=c.shape[-1])(c)

    return c


input_layer = Input(shape=(224, 224, 3))
x = tf.keras.layers.Rescaling(1./255)(input_layer)


x1 = branch(x, k=3, depth=5)
x2 = branch(x, k=5, depth=5)
x3 = branch(x, k=7, depth=5)


# Ablation options for multiscale feature fusion.
# concat = Multiply()([x1, x2, x3])
# concat = concatenate([x1, x2, x3])
concat = Add()([x1, x2, x3])


# Ablation options for post-fusion attention.
# concat = ChannelAttention()(concat)
# concat = SpatialAttention()(concat)
# concat = CBAMBlock()(concat)
# concat = SelfAttention(in_dim=concat.shape[-1])(concat)
concat = HybridAttention(in_dim=concat.shape[-1])(concat)


gap = GlobalAveragePooling2D()(concat)

fc = Dense(512, kernel_regularizer=regularizers.l2(1e-4))(gap)
fc = Activation('relu')(fc)
fc = Dropout(0.3)(fc)

fc = Dense(256, kernel_regularizer=regularizers.l2(1e-4))(fc)
fc = Activation('relu')(fc)
fc = Dropout(0.3)(fc)


output_layer = Dense(4, activation='softmax', dtype='float32')(fc)

model = Model(inputs=input_layer, outputs=output_layer, name=model_name)

print(model.name)
model.summary()


model.compile(
    optimizer=Adam(learning_rate=l_rate),
    loss=tf.keras.losses.CategoricalCrossentropy(),
    metrics=['accuracy', Precision(name="precision"), Recall(name="recall")]
)


info_file_path = os.path.join(output_dir, f"{model_name}_model_info.txt")

## Model Summary and Computational Profiling

This cell summarizes the HayabusaNet architecture and estimates its computational complexity. It stores the model summary as text, reports the total number of parameters, and calculates FLOPs using TensorFlow profiling.

In [ ]:
old_stdout = sys.stdout
buffer = StringIO()
sys.stdout = buffer
model.summary()
sys.stdout = old_stdout
summary_str = buffer.getvalue()

print(f"Total parameter: {model.count_params():,}")


try:
    input_spec = tf.TensorSpec([1, 224, 224, 3], tf.float32)
    full_model = tf.function(model).get_concrete_function(input_spec)
    frozen_func = convert_variables_to_constants_v2(full_model)
    graph = frozen_func.graph
    flops = profile(graph, options=ProfileOptionBuilder.float_operation())
    flops_str = f"FLOPs: {flops.total_float_ops:,} ({flops.total_float_ops / 1e9:.2f} GFLOPs)"
except Exception as e:
    flops_str = f"FLOPs calculation failed: {str(e)}"

print(flops_str)

## Training Callbacks

This cell defines the callbacks used during model training. The learning-rate scheduler reduces the learning rate when validation loss stops improving, while model checkpointing saves the best weights based on validation accuracy. A garbage-collection callback is also used to release unused Python objects after each epoch.

In [ ]:
from tensorflow.keras.callbacks import ReduceLROnPlateau, ModelCheckpoint


reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=8,
    verbose=1,
    mode='min',
    min_lr=1e-6
)

best_w_path = os.path.join(output_dir, f"{model_name}_best_weights.keras")
model_checkpoint = ModelCheckpoint(
    filepath=best_w_path,
    monitor='val_accuracy',
    save_best_only=True,
    save_weights_only=True,
    mode='max',
    verbose=1
)


class GarbageCollectionCallback(tf.keras.callbacks.Callback):
    """Run garbage collection after each training epoch.

    This callback helps release unused Python objects during notebook-based
    training and reports GPU memory usage when a GPU is available.
    """

    def on_epoch_end(self, epoch, logs=None):
        """Collect unused objects and report runtime memory status.

        Args:
            epoch: Index of the completed training epoch.
            logs: Optional dictionary of training logs.
        """
        gc.collect()

        gpu_devices = tf.config.list_physical_devices('GPU')
        if gpu_devices:
            try:
                memory_info = tf.config.experimental.get_memory_info('GPU:0')
                memory_gb = memory_info['current'] / 1024**3
                print(f"\nMemory after GC epoch {epoch + 1}: {memory_gb:.2f} GB")
            except (ValueError, RuntimeError):
                print(f"\nGarbage collection completed after epoch {epoch + 1}.")
        else:
            print(f"\nGarbage collection completed after epoch {epoch + 1}.")


gc_callback = GarbageCollectionCallback()

## Model Training

This cell trains HayabusaNet using the configured training and validation datasets. The training process applies learning-rate scheduling, best-weight checkpointing, and garbage collection after each epoch.

In [ ]:
start_time = time.time()

history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=EPOCH,
    callbacks=[reduce_lr, model_checkpoint, gc_callback],
    verbose=1
)

end_time = time.time()
elapsed_time = end_time - start_time
hours = int(elapsed_time // 3600)
minutes = int((elapsed_time % 3600) // 60)
seconds = int(elapsed_time % 60)

if hours > 0:
    training_time_str = f"Total training time: {hours}h {minutes}m {seconds}s"
elif minutes > 0:
    training_time_str = f"Total training time: {minutes}m {seconds}s"
else:
    training_time_str = f"Total training time: {seconds}s"

print(training_time_str)

## Training and Validation Curves

In [ ]:
acc1 = history.history['accuracy']
val_acc1 = history.history['val_accuracy']
loss1 = history.history['loss']
val_loss1 = history.history['val_loss']

plt.rcParams.update({'font.size': 11})
plt.style.use("seaborn-v0_8-whitegrid")

plt.figure()
plt.plot(acc1, '--', color='blue', label='Training Accuracy')
plt.plot(val_acc1, 'r', label='Validation Accuracy')
plt.ylabel('Accuracy')
plt.ylim([min(plt.ylim()), 1.01])
plt.title('Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.legend(loc='lower right')
plt.savefig(os.path.join(output_dir, f"{model_name}_val_accuracy.pdf"), format='pdf', dpi=300)

plt.figure()
plt.plot(loss1, '--', color='blue', label='Training Loss')
plt.plot(val_loss1, 'r', label='Validation Loss')
plt.legend(loc='upper right')
plt.ylabel('Cross Entropy')
plt.ylim([min(plt.ylim()), max(plt.ylim())])
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.savefig(os.path.join(output_dir, f"{model_name}_val_loss.pdf"), format='pdf', dpi=300)
plt.show()

## Validation Metrics, Confusion Matrix, ROC Curve, and Model Report

In [ ]:
model.load_weights(best_w_path)
best_model_path = os.path.join(output_dir, f"{model_name}_best_model.keras")
model.save(best_model_path)

results = model.evaluate(validation_generator, verbose=1)
print(f"Validation Loss: {results[0]:.4f}, Validation Accuracy: {results[1]:.4f}")

true_labels = []
for _, y in validation_generator:
    true_labels.extend(np.argmax(y, axis=1))
true_labels = np.array(true_labels)

class_labels = validation_generator.class_names

predictions = model.predict(validation_generator, verbose=1)
predicted_classes = np.argmax(predictions, axis=1)

conf_matrix = confusion_matrix(true_labels, predicted_classes)
conf_matrix_percent = conf_matrix.astype('float') / conf_matrix.sum(axis=1)[:, np.newaxis] * 100

annot_matrix = np.empty_like(conf_matrix).astype(str)
for i in range(conf_matrix.shape[0]):
    for j in range(conf_matrix.shape[1]):
        count = conf_matrix[i, j]
        percent = conf_matrix_percent[i, j]
        annot_matrix[i, j] = f"{count}\n({percent:.1f}%)"

plt.figure(figsize=(8, 6))
sns.heatmap(
    conf_matrix_percent,
    annot=annot_matrix,
    fmt='',
    cmap='Blues',
    xticklabels=class_labels,
    yticklabels=class_labels,
    cbar_kws={'label': 'Percentage (%)'}
)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.savefig(
    os.path.join(output_dir, f"{model_name}_confusion_matrix.pdf"),
    format='pdf',
    dpi=300
)
plt.show()


y_true_bin = label_binarize(true_labels, classes=range(len(class_labels)))

fpr = {}
tpr = {}
roc_auc = {}

for i in range(len(class_labels)):
    try:
        fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], predictions[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
    except ValueError:
        print(f"ROC curve could not be computed for class: {class_labels[i]}")

fpr["micro"], tpr["micro"], _ = roc_curve(y_true_bin.ravel(), predictions.ravel())
roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

plt.figure(figsize=(10, 8))
plt.plot(
    fpr["micro"],
    tpr["micro"],
    label=f'micro-average (AUC = {roc_auc["micro"]:.2f})',
    color='deeppink',
    linestyle=':',
    linewidth=4
)

colors = ['aqua', 'darkorange', 'cornflowerblue', 'green']
for i, color in zip(range(len(class_labels)), colors):
    if i in fpr:
        plt.plot(
            fpr[i],
            tpr[i],
            color=color,
            lw=2,
            label=f'{class_labels[i]} (AUC = {roc_auc[i]:.2f})'
        )

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curves')
plt.legend(loc="lower right")
plt.savefig(
    os.path.join(output_dir, f"{model_name}_ROC_curves.pdf"),
    format='pdf',
    dpi=300
)
plt.show()


accuracy = accuracy_score(true_labels, predicted_classes)
precision = precision_score(true_labels, predicted_classes, average='macro', zero_division=0)
recall = recall_score(true_labels, predicted_classes, average='macro', zero_division=0)
f1_val = f1_score(true_labels, predicted_classes, average='macro', zero_division=0)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1_val:.4f}")


with open(info_file_path, 'w') as f:
    f.write(f"Model Information: {model_name}\n")
    f.write("=" * 50 + "\n\n")

    f.write("MODEL SUMMARY:\n")
    f.write(summary_str)
    f.write("\n")

    f.write(f"Total parameters: {model.count_params():,}\n\n")

    f.write("COMPUTATIONAL COMPLEXITY:\n")
    f.write(flops_str + "\n\n")

    f.write("TRAINING INFORMATION:\n")
    f.write(training_time_str + "\n\n")

    f.write("FINAL VALIDATION METRICS — BEST CHECKPOINT BY VAL_ACCURACY:\n")
    f.write(f"Batch size: {BATCH_SIZE}\n")
    f.write(f"Epochs: {EPOCH}\n")
    f.write(f" - val_loss: {results[0]:.4f}\n")
    f.write(f" - val_acc: {accuracy:.4f}\n")
    f.write(f" - val_prec_macro: {precision:.4f}\n")
    f.write(f" - val_rec_macro: {recall:.4f}\n")
    f.write(f" - val_f1_macro: {f1_val:.4f}\n")

print(f"Model information saved to: {info_file_path}")
print(training_time_str)

## Inference Time Measurement

This cell measures the inference latency and throughput of the trained HayabusaNet model using the best saved checkpoint.

In [ ]:
infer_ds_bs1 = validation_generator.unbatch().batch(1)
infer_ds_bS = validation_generator


def measure_inference_time(model, ds, warmup=10, runs=50, desc="BS=1"):
    """Measure model inference latency and throughput.

    Args:
        model: Trained Keras model used for inference.
        ds: TensorFlow dataset used for inference timing.
        warmup: Number of initial batches excluded from timing.
        runs: Number of batches used for timing measurement.
        desc: Description of the inference setting.

    Returns:
        Dictionary containing latency and throughput statistics.
    """
    it = iter(ds)
    for _ in range(warmup):
        try:
            xb, _ = next(it)
        except StopIteration:
            break
        yb = model(xb, training=False)
        _ = yb.numpy()

    times = []
    n_samples = 0
    it = iter(ds)

    for _ in range(runs):
        try:
            xb, _ = next(it)
        except StopIteration:
            break

        bs = xb.shape[0]
        start = time.perf_counter()
        yb = model(xb, training=False)
        _ = yb.numpy()
        end = time.perf_counter()

        times.append(end - start)
        n_samples += bs

    if len(times) == 0:
        return {
            "desc": desc,
            "runs": 0,
            "avg_s": np.nan,
            "p50_s": np.nan,
            "p95_s": np.nan,
            "throughput": 0.0
        }

    times = np.array(times)
    avg_s = float(times.mean())
    p50_s = float(np.percentile(times, 50))
    p95_s = float(np.percentile(times, 95))

    total_time = float(times.sum())
    throughput = n_samples / total_time if total_time > 0 else 0.0

    return {
        "desc": desc,
        "runs": len(times),
        "avg_s": avg_s,
        "p50_s": p50_s,
        "p95_s": p95_s,
        "throughput": throughput
    }


model.load_weights(best_w_path)

res_bs1 = measure_inference_time(
    model,
    infer_ds_bs1,
    warmup=10,
    runs=100,
    desc="BS=1"
)

res_bSb = measure_inference_time(
    model,
    infer_ds_bS,
    warmup=3,
    runs=20,
    desc=f"BS={BATCH_SIZE}"
)

print(
    f"[Inference] {res_bs1['desc']}: avg={res_bs1['avg_s'] * 1e3:.2f} ms  "
    f"p50={res_bs1['p50_s'] * 1e3:.2f} ms  "
    f"p95={res_bs1['p95_s'] * 1e3:.2f} ms  "
    f"throughput≈{res_bs1['throughput']:.2f} img/s"
)

print(
    f"[Inference] {res_bSb['desc']}: avg/batch={res_bSb['avg_s'] * 1e3:.2f} ms  "
    f"p50={res_bSb['p50_s'] * 1e3:.2f} ms  "
    f"p95={res_bSb['p95_s'] * 1e3:.2f} ms  "
    f"throughput≈{res_bSb['throughput']:.2f} img/s"
)

p50_seconds = res_bs1["p50_s"]
p50_milliseconds = p50_seconds * 1e3

print(
    f"Single-number to report (p50, BS=1): "
    f"{p50_milliseconds:.2f} ms ({p50_seconds:.6f} s)"
)

avg_seconds = res_bs1["avg_s"]

print(
    f"Average per-image latency (BS=1): "
    f"{avg_seconds * 1e3:.2f} ms ({avg_seconds:.6f} s)"
)

with open(info_file_path, 'a') as f:
    f.write("\nINFERENCE LATENCY AND THROUGHPUT:\n")
    f.write(
        f" - {res_bs1['desc']}: avg={res_bs1['avg_s'] * 1e3:.2f} ms, "
        f"p50={res_bs1['p50_s'] * 1e3:.2f} ms, "
        f"p95={res_bs1['p95_s'] * 1e3:.2f} ms, "
        f"throughput≈{res_bs1['throughput']:.2f} img/s\n"
    )
    f.write(
        f" - {res_bSb['desc']}: avg/batch={res_bSb['avg_s'] * 1e3:.2f} ms, "
        f"p50={res_bSb['p50_s'] * 1e3:.2f} ms, "
        f"p95={res_bSb['p95_s'] * 1e3:.2f} ms, "
        f"throughput≈{res_bSb['throughput']:.2f} img/s\n"
    )

    f.write(
        f"\nSingle-number to report (p50 latency, BS=1): "
        f"{p50_milliseconds:.2f} ms ({p50_seconds:.6f} s)\n"
    )

    f.write(
        f"Average per-image latency (BS=1): "
        f"{avg_seconds * 1e3:.2f} ms ({avg_seconds:.6f} s)\n"
    )